In [ ]:
import numpy as np
from keras.datasets import mnist

Importing Dataset

In [ ]:
(X_train, Y_train), (X_test, Y_test) = mnist.load_data()
# sample 25 mnist digits from train dataset
indexes = np.random.randint(0, X_train.shape[0], size=25)

Flattening the data for training and applying one hot encoding & standarization

In [ ]:
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)

NUM_CLASSES = 10

Y_train = Y_train.reshape(Y_train.shape[0], -1)
Y_test = Y_test.reshape(Y_test.shape[0], -1)

def to_categorical_np(y, num_classes):
    return np.eye(num_classes)[y.reshape(-1)]

Y_train = to_categorical_np(Y_train, NUM_CLASSES)
Y_test = to_categorical_np(Y_test, NUM_CLASSES)


X_train = (X_train - np.mean(X_train)) / np.std(X_train)
X_test = (X_test - np.mean(X_test)) / np.std(X_test)

print(X_train.shape)
print(Y_train.shape)
print(X_test.shape)
print(Y_test.shape)

(60000, 784)
(60000, 10)
(10000, 784)
(10000, 10)


In [ ]:
print(X_train.min(), X_train.max())

-0.424073894391566 2.821543345689335


Activation Functions

In [ ]:
def sigmoid(Z):
    Z = np.clip(Z, -500, 500)  # Limit the range of Z to avoid overflow/underflow
    A = 1. / (1. + np.exp(-Z))
    return A


def derivative_sigmoid(Z):
    A = sigmoid(Z)
    return A * (1 - A)


def softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)  # numerical stability
    expZ = np.exp(z)
    return expZ / np.sum(expZ, axis=1, keepdims=True)

Initialize Parameters

In [ ]:
def initialize_parameters(layers_size):
    parameters = {}
    for l in range(1, len(layers_size)):
        parameters["W" + str(l)] = np.random.randn(layers_size[l], layers_size[l - 1]) / np.sqrt(layers_size[l - 1])
        parameters["b" + str(l)] = np.random.randn(layers_size[l], 1) / np.sqrt(layers_size[l - 1])
    return parameters


layer_dims = [X_train.shape[1], 20, 10, Y_train.shape[1]]
p = initialize_parameters(layer_dims)
for l in range(1, len(layer_dims)):
    print("shape of w" + str(l) + ":", p["W" + str(l)].shape)
    print("shape of b" + str(l) + ":", p['b' + str(l)].shape)

shape of w1: (20, 784)
shape of b1: (20, 1)
shape of w2: (10, 20)
shape of b2: (10, 1)
shape of w3: (10, 10)
shape of b3: (10, 1)


Forward Propagation

In [ ]:
def forward_propagation(X, Parameters):
    forward_cache = {}
    L = len(Parameters) // 2
    forward_cache["A0"] = X

    for l in range(1, L):
        Z = forward_cache["A" + str(l - 1)].dot(Parameters["W" + str(l)].T) + Parameters["b" + str(l)].T
        forward_cache["Z" + str(l)] = Z
        forward_cache["A" + str(l)] = sigmoid(Z)

    # LAST LAYER → SOFTMAX
    ZL = forward_cache["A" + str(L - 1)].dot(Parameters["W" + str(L)].T) + Parameters["b" + str(L)].T
    forward_cache["Z" + str(L)] = ZL
    forward_cache["A" + str(L)] = softmax(ZL)

    return forward_cache, forward_cache["A" + str(L)]


fcahce, A = forward_propagation(X_train, p)
for l in range(len(p) // 2 + 1):
    print(fcahce['A' + str(l)].shape)

(60000, 784)
(60000, 20)
(60000, 10)
(60000, 10)


In [ ]:
forw_cache, aL = forward_propagation(X_train, p)

for l in range(len(p) // 2 + 1):
    print("Shape of A" + str(l) + " :", forw_cache['A' + str(l)].shape)

Shape of A0 : (60000, 784)
Shape of A1 : (60000, 20)
Shape of A2 : (60000, 10)
Shape of A3 : (60000, 10)


Cost Function

In [ ]:
def compute_cost(AL, Y):
    eps = 1e-8
    loss = -np.mean(np.sum(Y * np.log(AL + eps), axis=1))
    return loss

Backward Propagation

In [ ]:
def backward_propagation(AL, Y, parameters, forward_cache):
    grads = {}
    L = len(parameters) // 2
    m = AL.shape[0]

    # OUTPUT LAYER
    grads["dZ" + str(L)] = AL - Y
    grads["dW" + str(L)] = (1 / m) * np.dot(grads["dZ" + str(L)].T,
                                            forward_cache["A" + str(L - 1)])
    grads["db" + str(L)] = (1 / m) * np.sum(grads["dZ" + str(L)], axis=0, keepdims=True)

    # HIDDEN LAYERS
    for l in reversed(range(1, L)):
        grads["dZ" + str(l)] = (
            np.dot(grads["dZ" + str(l + 1)], parameters["W" + str(l + 1)])
            * derivative_sigmoid(forward_cache["Z" + str(l)])
        )
        grads["dW" + str(l)] = (1 / m) * np.dot(grads["dZ" + str(l)].T,forward_cache["A" + str(l - 1)])
        grads["db" + str(l)] = (1 / m) * np.sum(grads["dZ" + str(l)], axis=0, keepdims=True)

    return grads


Update Parameters

In [ ]:
def update_parameters(parameters, grads, learning_rate):
    L = len(parameters) // 2

    for l in range(1, L + 1):
        parameters["W" + str(l)] = parameters["W" + str(l)] - learning_rate * grads["dW" + str(l)]
        parameters["b" + str(l)] -= learning_rate * grads["db" + str(l)].T


    return parameters

Predictions

In [ ]:
def predict(X, y, parameters):
    m = y.shape[0]
    _, y_pred = forward_propagation(X, parameters)

    # print(y_pred.shape)
    y = np.argmax(y, axis=1)
    y_pred = np.argmax(y_pred, axis=1)
    # print('\n', y_pred)
    # print('\n', y)
    return np.round((np.sum(y_pred == y) / m), 2)

Complete Model

In [ ]:
def NN(X, Y, num_of_layers, layer_dims):
    # np.random.seed(95)
    lr = 0.1
    iters = 101
    tot_layers = [784]
    for i in layer_dims:
        tot_layers.append(i)
    parameters = initialize_parameters(tot_layers)
    # grads = {}
    for i in range(0, iters):

        forward_cache, AL = forward_propagation(X, parameters)

        cost = compute_cost(AL, Y)

        grads = backward_propagation(AL, Y, parameters, forward_cache)

        parameters = update_parameters(parameters, grads, lr)
        if i % 10 == 0:
            print("\niter:{} \t cost: {} \t train_acc:{} \t test_acc:{}" .format(i, np.round(cost, 2),predict(X_train, Y_train, parameters) * 100,predict(X_test, Y_test, parameters) * 100))

        if i % 10 == 0:
            print("==", end='')
     # print(cost)
    return parameters

In [ ]:
parameters = NN(X_train, Y_train, 2, [500, 10])
acc = predict(X_test, Y_test, parameters)
print(acc * 100)


iter:0 	 cost: 2.52 	 train_acc:19.0 	 test_acc:20.0
==
iter:10 	 cost: 1.68 	 train_acc:72.0 	 test_acc:73.0
==
iter:20 	 cost: 1.29 	 train_acc:78.0 	 test_acc:79.0
==
iter:30 	 cost: 1.04 	 train_acc:81.0 	 test_acc:82.0
==
iter:40 	 cost: 0.89 	 train_acc:83.0 	 test_acc:84.0
==
iter:50 	 cost: 0.79 	 train_acc:84.0 	 test_acc:85.0
==
iter:60 	 cost: 0.71 	 train_acc:85.0 	 test_acc:86.0
==
iter:70 	 cost: 0.66 	 train_acc:86.0 	 test_acc:86.0
==
iter:80 	 cost: 0.61 	 train_acc:86.0 	 test_acc:87.0
==
iter:90 	 cost: 0.58 	 train_acc:86.0 	 test_acc:88.0
==
iter:100 	 cost: 0.55 	 train_acc:87.0 	 test_acc:88.0
==88.0
